# Full regularization sweep analyzer

Compare A recovery and control in abundant and single-shot.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 340)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root.")

REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"
SS_DIR = RESULTS_ROOT / "experiment_2026_09_29_single_shot_full_regularization_sweep"
AB_DIR = RESULTS_ROOT / "experiment_2026_09_29_abundant_full_regularization_sweep"
OUT_DIR = RESULTS_ROOT / "full_regularization_sweep_analysis_2026_09_29"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SS_PATH = SS_DIR / "single_shot_full_regularization_sweep_seed0.csv"
AB_PATH = AB_DIR / "regularization_sweep_all_runs.csv"
for p in [SS_PATH, AB_PATH]:
    if not p.exists(): raise FileNotFoundError(p)
print("Repo:", REPO_ROOT)


Repo: d:\Work\repos\RL\unknown_graph_networks


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_99460\221622189.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
ss = pd.read_csv(SS_PATH)
ab = pd.read_csv(AB_PATH)

ss_can = ss.rename(columns={
    "final_mean": "control",
    "final_fit_model_over_identity": "prediction_ratio",
}).copy()
ss_can["regime"] = "single_shot"

ab_can = ab.rename(columns={
    "mean_end": "control",
    "model_over_identity": "prediction_ratio",
}).copy()
ab_can["regime"] = "abundant"

metrics = [
    "control", "prediction_ratio", "A_MAE", "A_Fro", "row_TV",
    "true_edge_mass", "support_topdegree_recall", "edge_AP",
    "centrality_L1", "row_entropy_norm", "effective_neighbors",
]
keep = [
    "regime", "environment_id", "scenario_class", "family", "dynamics",
    "regularizer_kind", "regularizer_lambda", *metrics,
]
work = pd.concat([ss_can[keep], ab_can[keep]], ignore_index=True)

counts = work.groupby(["regime","regularizer_kind","regularizer_lambda"]).size()
assert counts.eq(195).all(), counts[counts != 195]
print("Validated all conditions at 195 environments each.")
display(counts.rename("n").reset_index())


Validated all conditions at 195 environments each.


,regime,regularizer_kind,regularizer_lambda,n
0,abundant,entropy,1.000000e-08,195
1,abundant,entropy,1.000000e-07,195
2,abundant,entropy,3.000000e-07,195
3,abundant,entropy,1.000000e-06,195
4,abundant,entropy,3.000000e-06,195
5,abundant,entropy,1.000000e-05,195
6,abundant,entropy,3.000000e-05,195
7,abundant,entropy,1.000000e-04,195
8,abundant,entropy,3.000000e-04,195
9,abundant,entropy,1.000000e-03,195


In [3]:
refs = (
    work[(work["regularizer_kind"] == "none") & (work["regularizer_lambda"] == 0.0)]
    .set_index(["regime","environment_id"])
)
assert len(refs) == 390

for metric in metrics:
    work[f"delta_{metric}"] = work.apply(
        lambda r: float(r[metric]) - float(refs.loc[(r["regime"],r["environment_id"]), metric]),
        axis=1,
    )

summary = (
    work.groupby(["regime","regularizer_kind","regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id","size"),
        control=("control","mean"),
        delta_control=("delta_control","mean"),
        prediction_ratio=("prediction_ratio","mean"),
        delta_prediction=("delta_prediction_ratio","mean"),
        A_MAE=("A_MAE","mean"),
        delta_A_MAE=("delta_A_MAE","mean"),
        row_TV=("row_TV","mean"),
        delta_row_TV=("delta_row_TV","mean"),
        true_edge_mass=("true_edge_mass","mean"),
        delta_true_edge_mass=("delta_true_edge_mass","mean"),
        edge_AP=("edge_AP","mean"),
        delta_edge_AP=("delta_edge_AP","mean"),
        centrality_L1=("centrality_L1","mean"),
        delta_centrality_L1=("delta_centrality_L1","mean"),
        row_entropy_norm=("row_entropy_norm","mean"),
        delta_row_entropy=("delta_row_entropy_norm","mean"),
        effective_neighbors=("effective_neighbors","mean"),
        delta_effective_neighbors=("delta_effective_neighbors","mean"),
    )
)

by_dyn = (
    work.groupby(["regime","regularizer_kind","regularizer_lambda","scenario_class","dynamics"], as_index=False)
    .agg(
        n=("environment_id","size"),
        delta_control=("delta_control","mean"),
        delta_prediction=("delta_prediction_ratio","mean"),
        delta_row_TV=("delta_row_TV","mean"),
        delta_true_edge_mass=("delta_true_edge_mass","mean"),
        delta_edge_AP=("delta_edge_AP","mean"),
        delta_centrality_L1=("delta_centrality_L1","mean"),
        delta_effective_neighbors=("delta_effective_neighbors","mean"),
    )
)

worst = (
    by_dyn.groupby(["regime","regularizer_kind","regularizer_lambda"], as_index=False)
    .agg(
        worst_dynamic_control_delta=("delta_control","min"),
        worst_dynamic_row_TV_delta=("delta_row_TV","max"),
        worst_dynamic_prediction_delta=("delta_prediction","max"),
    )
)
summary = summary.merge(worst, on=["regime","regularizer_kind","regularizer_lambda"], how="left")
summary.to_csv(OUT_DIR / "full_regularization_summary.csv", index=False)
by_dyn.to_csv(OUT_DIR / "full_regularization_by_dynamics.csv", index=False)
display(summary.round(6))


,regime,regularizer_kind,regularizer_lambda,n,control,delta_control,prediction_ratio,delta_prediction,A_MAE,delta_A_MAE,row_TV,delta_row_TV,true_edge_mass,delta_true_edge_mass,edge_AP,delta_edge_AP,centrality_L1,delta_centrality_L1,row_entropy_norm,delta_row_entropy,effective_neighbors,delta_effective_neighbors,worst_dynamic_control_delta,worst_dynamic_row_TV_delta,worst_dynamic_prediction_delta
0,abundant,entropy,0.000000,195,0.768628,0.000000,0.638309,-0.000001,0.076799,-0.000000,0.575995,-0.000002,0.433525,0.000002,0.741847,-0.000000,0.480079,-0.000002,0.903401,-0.000002,9.154730,-0.000097,0.000000,0.000000,-0.000000
1,abundant,entropy,0.000000,195,0.768635,0.000007,0.638302,-0.000008,0.076797,-0.000002,0.575979,-0.000017,0.433543,0.000020,0.741846,-0.000001,0.480061,-0.000020,0.903381,-0.000022,9.153857,-0.000970,0.000000,0.000002,-0.000000
2,abundant,entropy,0.000000,195,0.768664,0.000036,0.638285,-0.000025,0.076793,-0.000007,0.575944,-0.000052,0.433582,0.000059,0.741844,-0.000003,0.480021,-0.000060,0.903338,-0.000065,9.151914,-0.002912,0.000000,0.000005,-0.000000
3,abundant,entropy,0.000001,195,0.768662,0.000034,0.638227,-0.000084,0.076776,-0.000023,0.575823,-0.000173,0.433720,0.000197,0.741728,-0.000120,0.479880,-0.000201,0.903186,-0.000217,9.145097,-0.009730,-0.000000,0.000016,-0.000001
4,abundant,entropy,0.000003,195,0.768531,-0.000097,0.638060,-0.000250,0.076731,-0.000069,0.575479,-0.000517,0.434112,0.000589,0.741432,-0.000415,0.479479,-0.000602,0.902748,-0.000655,9.125453,-0.029374,-0.000839,0.000051,-0.000002
5,abundant,entropy,0.000010,195,0.768169,-0.000458,0.637492,-0.000818,0.076574,-0.000225,0.574308,-0.001688,0.435473,0.001950,0.740164,-0.001683,0.478098,-0.001984,0.901170,-0.002233,9.054728,-0.100099,-0.003301,0.000195,-0.000005
6,abundant,entropy,0.000030,195,0.768182,-0.000445,0.635996,-0.002315,0.076171,-0.000628,0.571285,-0.004711,0.439201,0.005678,0.735306,-0.006541,0.474317,-0.005764,0.896265,-0.007138,8.835373,-0.319453,-0.005754,0.000730,0.000000
7,abundant,entropy,0.000100,195,0.765836,-0.002792,0.633075,-0.005236,0.075481,-0.001318,0.566109,-0.009887,0.448689,0.015165,0.712357,-0.029491,0.465901,-0.014180,0.874409,-0.028994,7.925858,-1.228968,-0.018908,0.004546,0.000300
8,abundant,entropy,0.000300,195,0.760158,-0.008469,0.641671,0.003361,0.077577,0.000778,0.581831,0.005834,0.445209,0.011686,0.643862,-0.097985,0.478249,-0.001832,0.813164,-0.090239,6.282098,-2.872729,-0.026891,0.041250,0.035173
9,abundant,entropy,0.001000,195,0.745212,-0.023416,0.689548,0.051238,0.087360,0.010561,0.655201,0.079204,0.381751,-0.051773,0.488856,-0.252991,0.536793,0.056711,0.750924,-0.152479,5.462997,-3.691830,-0.087144,0.239625,0.200007


In [4]:
metrics_wide = [
    "delta_control", "delta_prediction", "delta_A_MAE", "delta_row_TV",
    "delta_true_edge_mass", "delta_edge_AP", "delta_centrality_L1",
    "delta_effective_neighbors", "worst_dynamic_control_delta",
    "worst_dynamic_row_TV_delta",
]
parts=[]
for regime in ["abundant","single_shot"]:
    p = summary[summary["regime"] == regime][
        ["regularizer_kind","regularizer_lambda", *metrics_wide]
    ].copy()
    p = p.rename(columns={c:f"{regime}_{c}" for c in metrics_wide})
    parts.append(p)
joint = parts[0].merge(parts[1], on=["regularizer_kind","regularizer_lambda"], validate="one_to_one")
joint = joint.sort_values(["regularizer_kind","regularizer_lambda"])
joint.to_csv(OUT_DIR / "full_regularization_joint_side_by_side.csv", index=False)
print("Deltas versus no regularization")
display(joint.round(6))


Deltas versus no regularization


,regularizer_kind,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_A_MAE,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_delta_effective_neighbors,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_A_MAE,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_delta_effective_neighbors,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_row_TV_delta
0,entropy,0.000000,0.000000,-0.000001,-0.000000,-0.000002,0.000002,-0.000000,-0.000002,-0.000097,0.000000,0.000000,0.000028,-0.000107,0.000039,0.000294,-0.000081,-0.000173,0.000512,-0.006331,-0.000154,0.002125
1,entropy,0.000000,0.000007,-0.000008,-0.000002,-0.000017,0.000020,-0.000001,-0.000020,-0.000970,0.000000,0.000002,0.000061,0.000814,0.000168,0.001264,-0.000386,-0.001943,0.004188,-0.060786,-0.000646,0.012806
2,entropy,0.000000,0.000036,-0.000025,-0.000007,-0.000052,0.000059,-0.000003,-0.000060,-0.002912,0.000000,0.000005,-0.000054,0.002545,0.000407,0.003050,0.000205,-0.005074,0.008715,-0.179795,-0.002983,0.031242
3,entropy,0.000001,0.000034,-0.000084,-0.000023,-0.000173,0.000197,-0.000120,-0.000201,-0.009730,-0.000000,0.000016,-0.000154,0.007427,0.001341,0.010059,0.003207,-0.014394,0.031355,-0.578327,-0.003596,0.072631
4,entropy,0.000003,-0.000097,-0.000250,-0.000069,-0.000517,0.000589,-0.000415,-0.000602,-0.029374,-0.000839,0.000051,-0.001791,0.025722,0.003055,0.022913,0.020277,-0.035228,0.105202,-1.241300,-0.012566,0.119244
5,entropy,0.000010,-0.000458,-0.000818,-0.000225,-0.001688,0.001950,-0.001683,-0.001984,-0.100099,-0.003301,0.000195,-0.007618,0.077765,0.007749,0.058118,0.024679,-0.078079,0.224850,-1.880013,-0.028787,0.227242
6,entropy,0.000030,-0.000445,-0.002315,-0.000628,-0.004711,0.005678,-0.006541,-0.005764,-0.319453,-0.005754,0.000730,-0.022879,0.154788,0.016507,0.123803,-0.009802,-0.144803,0.357737,-2.216890,-0.049566,0.266339
7,entropy,0.000100,-0.002792,-0.005236,-0.001318,-0.009887,0.015165,-0.029491,-0.014180,-1.228968,-0.018908,0.004546,-0.044256,0.295252,0.031085,0.233139,-0.073235,-0.219263,0.561233,-2.557591,-0.161463,0.556562
8,entropy,0.000300,-0.008469,0.003361,0.000778,0.005834,0.011686,-0.097985,-0.001832,-2.872729,-0.026891,0.041250,-0.066542,0.385525,0.041360,0.310200,-0.128871,-0.270701,0.729127,-2.717837,-0.223392,0.708968
9,entropy,0.001000,-0.023416,0.051238,0.010561,0.079204,-0.051773,-0.252991,0.056711,-3.691830,-0.087144,0.239625,-0.082758,0.453492,0.051658,0.387434,-0.200136,-0.317838,0.873389,-2.827104,-0.250773,0.807101


In [5]:
MAX_CONTROL_DROP = 0.001
candidates = joint[
    (joint["regularizer_kind"] != "none")
    & (joint["abundant_delta_control"] >= -MAX_CONTROL_DROP)
    & (joint["single_shot_delta_control"] >= -MAX_CONTROL_DROP)
    & (joint["abundant_delta_row_TV"] < 0)
    & (joint["single_shot_delta_row_TV"] < 0)
].copy()

print(f"Candidates with row-TV improvement in both regimes and control drop <= {MAX_CONTROL_DROP:g}:")
display(candidates.round(6))

print("\nDiffusion check (effective neighbors; negative means more concentrated):")
display(joint[[
    "regularizer_kind","regularizer_lambda",
    "abundant_delta_effective_neighbors","single_shot_delta_effective_neighbors",
    "abundant_delta_row_TV","single_shot_delta_row_TV",
    "abundant_delta_control","single_shot_delta_control",
]].round(6))

print("\nOutputs:", OUT_DIR)
print("FULL REGULARIZATION SWEEP ANALYSIS COMPLETE")


Candidates with row-TV improvement in both regimes and control drop <= 0.001:


,regularizer_kind,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_A_MAE,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_delta_effective_neighbors,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_A_MAE,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_delta_effective_neighbors,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_row_TV_delta



Diffusion check (effective neighbors; negative means more concentrated):


,regularizer_kind,regularizer_lambda,abundant_delta_effective_neighbors,single_shot_delta_effective_neighbors,abundant_delta_row_TV,single_shot_delta_row_TV,abundant_delta_control,single_shot_delta_control
0,entropy,0.000000,-0.000097,-0.006331,-0.000002,0.000294,0.000000,0.000028
1,entropy,0.000000,-0.000970,-0.060786,-0.000017,0.001264,0.000007,0.000061
2,entropy,0.000000,-0.002912,-0.179795,-0.000052,0.003050,0.000036,-0.000054
3,entropy,0.000001,-0.009730,-0.578327,-0.000173,0.010059,0.000034,-0.000154
4,entropy,0.000003,-0.029374,-1.241300,-0.000517,0.022913,-0.000097,-0.001791
5,entropy,0.000010,-0.100099,-1.880013,-0.001688,0.058118,-0.000458,-0.007618
6,entropy,0.000030,-0.319453,-2.216890,-0.004711,0.123803,-0.000445,-0.022879
7,entropy,0.000100,-1.228968,-2.557591,-0.009887,0.233139,-0.002792,-0.044256
8,entropy,0.000300,-2.872729,-2.717837,0.005834,0.310200,-0.008469,-0.066542
9,entropy,0.001000,-3.691830,-2.827104,0.079204,0.387434,-0.023416,-0.082758



Outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\full_regularization_sweep_analysis_2026_09_29
FULL REGULARIZATION SWEEP ANALYSIS COMPLETE
